## Week 2 – Exercise – Individual – Data Quality Audit
### 100006646 - Arya Shinde

### **Dataset Description & Business use case:**
DESCRIPTION: The dataset contains 11 transaction records (including one duplicate) with attributes: transaction_id, customer_id, transaction_date, amount, currency, payment_method, status, region, and last_updated.

BUSINESS USE CASE: This data is likely used for Financial Auditing and Revenue Recognition. In a real-world scenario, this data helps track regional sales performance and payment method preferences. However, the current "messiness" would lead to incorrect tax calculations and skewed KPIs.

In [8]:
# import libraries
import pandas as pd
import numpy as np

In [13]:
# Read the csv file
df = pd.read_csv('../Data/week2_customer_transactions_messy.csv')

# Initial scan
print(df.head(5))
print(df.info())
print(df.describe())

  transaction_id customer_id transaction_date  amount currency payment_method  \
0          T0001        C100       2026-01-05  120.50      EUR           card   
1          T0002        C101       2026/01/06    0.00      EUR           CARD   
2          T0003        C102       06-01-2026  -35.00      USD  bank_transfer   
3          T0004         NaN       2026-01-07  250.00      EUR           card   
4          T0005        C104       2026-01-07   89.99     EURO           cash   

      status region last_updated  
0  completed     DE   2026-01-05  
1  completed     de   2026-01-20  
2  completed     US   2026-01-07  
3    pending     FR   2026-01-08  
4  completed     DE   2026-01-09  
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    11 non-null     str    
 1   customer_id       10 non-null     str    
 2   transaction_date  1

### **Issues by Dimension**

In [ ]:
issue_table = pd.DataFrame([
    ['Missing amount/ID', 'Completeness', 'Underestimates total revenue and LTV calculations.'],
    ['Duplicate rows', 'Uniqueness', 'Leads to double-counting of sales and inflated KPIs.'],
    ['Outlier/Negative amounts', 'Validity', 'Indicates data entry errors or unhandled refund logic.'],
    ['Category formatting', 'Consistency', 'Prevents accurate grouping of sales by department.']
], columns=['Issue', 'Dimension', 'Impact'])
issue_table

,Issue,Dimension,Impact
0,Missing amount/ID,Completeness,Underestimates total revenue and LTV calculati...
1,Duplicate rows,Uniqueness,Leads to double-counting of sales and inflated...
2,Outlier/Negative amounts,Validity,Indicates data entry errors or unhandled refun...
3,Category formatting,Consistency,Prevents accurate grouping of sales by departm...


### **KPI Calculations**

In [ ]:
# Completeness Rate (Customer ID and Amount)
missing_count = df[['customer_id', 'amount']].isnull().any(axis=1).sum()
completeness_rate = ((len(df) - missing_count) / len(df)) * 100

# Duplication Rate
dup_count = df.duplicated().sum()
duplication_rate = (dup_count / len(df)) * 100

# Validity Error Rate (Amount > 0 and realistic)
valid_amount = df['amount'].apply(lambda x: 0 < x < 1000000 if pd.notnull(x) else False)
error_rate = ((len(df) - valid_amount.sum()) / len(df)) * 100

print(f"KPI Summary:")
print(f"Completeness Rate: {completeness_rate:.2f}%")
print(f"Duplication Rate: {duplication_rate:.2f}%")
print(f"Error Rate (Validity): {error_rate:.2f}%")

KPI Summary:
Completeness Rate: 81.82%
Duplication Rate: 9.09%
Error Rate (Validity): 36.36%


### **KPI Interpretation**
**Completeness Rate (81.82%):**

Meaning: Approximately 18% of the records are missing either a customer_id or an amount.

Impact: This is a high rate of missing data. In a financial context, this represents "revenue leakage" - transactions occurring that cannot be attributed to a specific customer or valued correctly.

**Duplication Rate (9.09%):**

Meaning: About 9% of the rows in the dataset are exact duplicates of other rows.

Impact: This suggests an issue with the data ingestion pipeline (a script running twice or a lack of unique constraints). It will result in over-reporting sales figures by nearly 10%.

**Error Rate / Validity (36.36%):**

Meaning: Over one-third of the data fails the "realistic amount" check (either it's non-numeric, negative, or exceeds $1,000,000).

Impact: This is the most severe finding. It indicates that the amount column is highly untrustworthy and likely requires extensive cleaning or a "hard reset" on how that data is being collected.

### **Validation rules**

In [ ]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T

,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


### **Short Audit Summary**

In [ ]:
# Create a summary based on the actual audit results
audit_data = [
    ['Invalid/Outlier Amounts', '36.36%', 'Critical', 'Verify values > $1M and handle negative entries.'],
    ['Missing ID or Amount', '18.18%', 'High', 'Isolate rows missing customer_id for reconciliation.'],
    ['Duplicate Transactions', '9.09%', 'Medium', 'Apply drop_duplicates() to stabilize revenue reporting.']
]

audit_summary = pd.DataFrame(audit_data, columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])
audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Invalid/Outlier Amounts,36.36%,Critical,Verify values > $1M and handle negative entries.
1,Missing ID or Amount,18.18%,High,Isolate rows missing customer_id for reconcili...
2,Duplicate Transactions,9.09%,Medium,Apply drop_duplicates() to stabilize revenue r...


### **Recommended Cleaning Actions for next week:**

| Step | Action Item | Priority | Expected Outcome |
| :--- | :--- | :--- | :--- |
| **1** | **Schema Normalization** | **Critical** | Use `.str.strip().lower()` on all headers to eliminate `KeyErrors` and trailing spaces. |
| **2** | **Record Deduplication** | **High** | Apply `drop_duplicates()` to remove the 9.09% redundant rows and ensure unique `transaction_id`s. |
| **3** | **Financial Type Casting** | **Critical** | Convert `amount` to numeric and `transaction_date` to datetime with `errors='coerce'` to isolate unparseable strings. |
| **4** | **Strategic Imputation** | **Medium** | Fill missing `amount` values with the median of their respective `product_category` to preserve distribution. |
| **5** | **Label Standardization** | **Low** | Use a mapping dictionary (e.g., `{'electr': 'Electronics'}`) to fix the inconsistent categorical labels. |